# TxGNN Inference Pipeline

 inference pipeline for TxGNN



## Pipeline Overview

1. **Data Loading** — Load the biomedical knowledge graph via `TxData`
2. **Train/Test Split** — Prepare the complex disease split (zero-shot setting)
3. **Model Initialization** — Configure the HeteroRGCN + DistMult + Metric Learning architecture
4. **Pre-training** — Link prediction pre-training across all 30+ edge types
5. **Fine-tuning** — Drug-disease relation fine-tuning with prototype-enhanced metric learning
6. **Model Saving/Loading** — Checkpoint management
7. **Disease-Centric Evaluation** — Per-disease ranking of drug candidates
8. **Inference on Specific Diseases** — Zero-shot drug repurposing predictions
9. **Embedding Retrieval & Visualization** — Node embedding analysis
10. **GraphMask Explainability** — Post-hoc explanation of predictions via meta-paths

## 0. Environment Setup

The PyPI package for TxGNN is broken — `pip install TxGNN` fails because `requirements.txt` is missing from the sdist ([issue #1](https://github.com/mims-harvard/TxGNN/issues/1), [issue #4](https://github.com/mims-harvard/TxGNN/issues/4)).

### Data and Code Access
We will use a local clone of the TxGNN repository stored in Google Drive. This allows us to modify `model.py` if needed for custom data and ensure we are using the latest version of the code.

### Setup Steps
1.  **Mount Google Drive** to access the project folder.
2.  **Add the repository path to `PYTHONPATH`** so Python can import `txgnn` directly from the folder.
3.  **Install compatible dependencies** if necessary.



In [2]:
# mount to gdrive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702
702_Rare_Disease.ipynb		 README.md
condacolab_install.log		 src
method_comparison.png		 TxGNN
phenotype_drug_experiment.ipynb  TxGNN_Inference_Pipeline.ipynb
phenotype_drug_TxGNN.ipynb


In [3]:
import os, sys

# ── Project paths ───────────────────────────────────────────────────────────
project_path    = os.getcwd()
txgnn_repo_path = os.path.join(project_path, 'TxGNN')

# Clone from GitHub if the local copy is not already present
if not os.path.isdir(os.path.join(txgnn_repo_path, 'txgnn')):
    print("Cloning TxGNN repository from GitHub …")
    !git clone https://github.com/mims-harvard/TxGNN.git "{txgnn_repo_path}"
else:
    print(f"Using existing TxGNN repo at: {txgnn_repo_path}")

# Add the repo root to PYTHONPATH so the 'txgnn' sub-package is importable
# without a pip install (avoids the broken PyPI sdist — issues #1, #4)
os.environ['PYTHONPATH'] = txgnn_repo_path + ':' + os.environ.get('PYTHONPATH', '')
if txgnn_repo_path not in sys.path:
    sys.path.insert(0, txgnn_repo_path)

print(f"PYTHONPATH updated — '{txgnn_repo_path}' is now on the path")


Using existing TxGNN repo at: /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/TxGNN
PYTHONPATH updated — '/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/TxGNN' is now on the path


In [ ]:
# Install compatible dependencies
# The PyPI TxGNN package is broken (issues #1, #4) — we use the cloned repo via PYTHONPATH.
# NOTE: After this cell finishes, do Runtime → Restart session, then re-run from cell 3.

import subprocess, sys

# 1. Pin NumPy < 2 — DGL and older torch C-extensions were compiled against NumPy 1.x.
#    NumPy 2.x breaks the torch↔numpy bridge (_ARRAY_API not found warning → downstream crashes).
!pip install "numpy<2" -q

# 2. DGL (latest wheel)
!pip install dgl -q

# 3. Detect installed torch version and pick the matching torchdata
#    torchdata 0.7.x → torch 2.2.x | 0.8.x → torch 2.3–2.4.x | 0.9.x → torch 2.5+
result = subprocess.run([sys.executable, '-c',
    'import torch; v=torch.__version__; '
    'major,minor=v.split(".")[:2]; print(major,minor)'],
    capture_output=True, text=True)
major, minor = (int(x) for x in result.stdout.strip().split())
print(f"Detected torch {major}.{minor}")

if (major, minor) >= (2, 5):
    torchdata_pin = ""
elif (major, minor) >= (2, 3):
    torchdata_pin = "==0.8.0"
elif (major, minor) >= (2, 2):
    torchdata_pin = "==0.7.1"
else:
    torchdata_pin = "==0.6.0"

print(f"Installing torchdata{torchdata_pin}")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'torchdata{torchdata_pin}'], check=False)

# 4. Other dependencies
!pip install pyyaml pydantic pandas -q



Detected torch 2.2
Installing torchdata==0.7.1

Done. ⚠️  Restart the runtime now (Runtime → Restart session), then re-run from cell 3.


---
## 1. Data Loading — Knowledge Graph Construction

TxGNN operates on a comprehensive biomedical knowledge graph containing **17,080 diseases** and **7,957 drugs**, connected through **30+ relation types** (disease-protein, drug-protein, disease-phenotype, drug-drug interactions, etc.).

`TxData` handles:
- Downloading raw KG data from Harvard Dataverse (if not cached)
- Preprocessing into a directed heterogeneous graph
- Creating train/valid/test splits per the specified evaluation protocol

In [4]:
import os, sys, types

# ── Compatibility shim ────────────────────────────────────────────────────────
# torch.utils._import_utils was removed in PyTorch >= 2.4.
# torchdata's datapipes (used internally by DGL graphbolt) still references it.
# Create a stub module so the import chain doesn't break.
import torch.utils as _tu
if 'torch.utils._import_utils' not in sys.modules:
    _stub = types.ModuleType('torch.utils._import_utils')
    _stub.dill_available = lambda: False
    sys.modules['torch.utils._import_utils'] = _stub
    _tu._import_utils = _stub

from txgnn import TxData, TxGNN, TxEval

# Data lives inside the cloned TxGNN repo
DATA_DIR = os.path.join(txgnn_repo_path, 'data')
print(f"Using data directory: {DATA_DIR}")

# Initialize TxData
TxData_obj = TxData(data_folder_path=DATA_DIR)


Using data directory: /content/drive/MyDrive/BMI Year1/BMI702/project/BMI702/TxGNN/data
Found local copy...
Found local copy...
Found local copy...


---
## 2. Train/Test Split — Complex Disease Split (Zero-Shot Setting)

The **complex disease split** (`split='complex_disease'`) is the primary evaluation protocol in the paper:

1. Sample a set of diseases from the KG
2. Move **all** of their drug-disease edges (indication, contraindication, off-label use) to the test set
3. These diseases have **zero treatments** in training — testing the model's zero-shot capability

Other available splits:
- `'random'` — random shuffle (most diseases seen during training)
- Disease-area splits: `'cell_proliferation'`, `'mental_health'`, `'cardiovascular'`, `'anemia'`, `'adrenal_gland'`, `'autoimmune'`, `'metabolic_disorder'`, `'diabetes'`, `'neurodigenerative'`
- `'full_graph'` — deploy mode with 95% train / 5% validation, no held-out test
- `'disease_eval'` — evaluate a single specific disease

In [5]:
# Prepare the complex disease split (zero-shot evaluation setting)
TxData_obj.prepare_split(
    split='complex_disease',
    seed=42,
    no_kg=False  # Set True for ablation without KG context
)

# Inspect split statistics
print(f"\n=== Split Statistics ===")
print(f"Full KG edges:      {len(TxData_obj.df):,}")
print(f"Training edges:     {len(TxData_obj.df_train):,}")
print(f"Validation edges:   {len(TxData_obj.df_valid):,}")
print(f"Test edges:         {len(TxData_obj.df_test):,}")
print(f"\nGraph node types:   {TxData_obj.G.ntypes}")
print(f"Graph edge types:   {len(TxData_obj.G.canonical_etypes)}")
print(f"Total nodes:        {TxData_obj.G.number_of_nodes():,}")
print(f"Total edges:        {TxData_obj.G.number_of_edges():,}")

Found saved processed KG... Loading...
Splits detected... Loading splits....
Creating DGL graph....
Done!

=== Split Statistics ===
Full KG edges:      4,050,249
Training edges:     6,735,722
Validation edges:   959,334
Test edges:         405,442

Graph node types:   ['anatomy', 'biological_process', 'cellular_component', 'disease', 'drug', 'effect/phenotype', 'exposure', 'gene/protein', 'molecular_function', 'pathway']
Graph edge types:   50
Total nodes:        129,312
Total edges:        6,735,722


In [6]:
# Examine the drug-disease edge distribution in the test set
dd_rel_types = ['indication', 'contraindication', 'off-label use']
df_test_dd = TxData_obj.df_test[TxData_obj.df_test.relation.isin(dd_rel_types)]

print("=== Test Set Drug-Disease Edges ===")
for rel in dd_rel_types:
    count = len(df_test_dd[df_test_dd.relation == rel])
    n_diseases = df_test_dd[df_test_dd.relation == rel].y_idx.nunique()
    print(f"  {rel:20s}: {count:5d} edges across {n_diseases} diseases")

print(f"\nTotal zero-shot test diseases: {df_test_dd.y_idx.nunique()}")

=== Test Set Drug-Disease Edges ===
  indication          :   495 edges across 66 diseases
  contraindication    :  1729 edges across 54 diseases
  off-label use       :   116 edges across 35 diseases

Total zero-shot test diseases: 103


---
## 3. Model Architecture Initialization

TxGNN consists of three components:

### A. Heterogeneous Graph Neural Network (HeteroRGCN)
A 2-layer Relational Graph Convolutional Network that learns node embeddings via message passing over the heterogeneous KG. Each layer applies per-relation linear transformations:
$$h_v^{(l+1)} = \sigma\left(\sum_{r \in \mathcal{R}} \sum_{u \in \mathcal{N}_r(v)} \frac{1}{|\mathcal{N}_r(v)|} W_r^{(l)} h_u^{(l)}\right)$$

### B. DistMult Decoder
Scores drug-disease pairs using a bilinear function:
$$\text{score}(u, v, r) = \sum_i h_u^{(i)} \cdot W_r^{(i)} \cdot h_v^{(i)}$$

### C. Metric Learning Module (TxGNN's Key Innovation)
For zero-shot diseases, the model retrieves `proto_num` similar diseases from the KG and augments the target disease embedding with prototype embeddings:
$$\tilde{h}_d = g \cdot h_d + (1 - g) \cdot \sum_{k=1}^{K} \alpha_k h_{d_k}$$
where $g$ is a learned gate, $\alpha_k$ are similarity-based coefficients, and $h_{d_k}$ are the embeddings of similar diseases.

In [ ]:
#TODO: FIX MODEL INIT ERROR

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Initialize the TxGNN model
model = TxGNN(
    data=TxData_obj,
    weight_bias_track=False,   # Set True to log to W&B
    proj_name='TxGNN',
    exp_name='TxGNN_inference',
    device=DEVICE
)

# Model hyperparameters (as in the paper)
model.model_initialize(
    n_hid=100,                    # Hidden dimension
    n_inp=100,                    # Input embedding dimension
    n_out=100,                    # Output dimension
    proto=True,                   # Enable metric learning module
    proto_num=3,                  # Number of prototype diseases to retrieve
    attention=False,              # Attention layer (False for GraphMask compatibility)
    sim_measure='all_nodes_profile',  # Disease similarity: 'all_nodes_profile', 'protein_profile', 'protein_random_walk'
    bert_measure='disease_name',       # BERT-based similarity measure
    agg_measure='rarity',              # Aggregation: 'rarity' (degree-based), 'avg', 'learn'
    num_walks=200,                     # Random walk count (for protein_random_walk)
    walk_mode='bit',                   # Walk encoding mode
    path_length=2                      # Path length for random walks
)

# Print model summary
n_params = sum(p.numel() for p in model.model.parameters())
print(f"\nModel initialized with {n_params:,} parameters")
print(f"Config: {model.config}")

---
## 4. Pre-training — Multi-Relational Link Prediction

Pre-training optimizes the GNN embeddings via link prediction across **all** edge types in the KG (not just drug-disease). This teaches the model the global structure of the biomedical knowledge graph.

- Uses mini-batch training with `dgl.dataloading.EdgeDataLoader`
- Negative sampling via destination corruption (`fix_dst`)
- Binary cross-entropy loss

**Note:** Pre-training is computationally expensive. For inference with pre-trained weights, skip to Section 6.

In [ ]:
# Pre-training (skip if loading pretrained model)
# In the paper: n_epoch=2, learning_rate=1e-3, batch_size=1024

TRAIN_FROM_SCRATCH = False  # Set True to train from scratch

if TRAIN_FROM_SCRATCH:
    model.pretrain(
        n_epoch=2,
        learning_rate=1e-3,
        batch_size=1024,
        train_print_per_n=20
    )
else:
    print("Skipping pre-training — will load pretrained model in Section 6.")

---
## 5. Fine-tuning — Drug-Disease Relation Learning with Metric Learning

Fine-tuning focuses specifically on the 6 drug-disease edge types:
- `(drug, indication, disease)` and reverse
- `(drug, contraindication, disease)` and reverse  
- `(drug, off-label use, disease)` and reverse

The metric learning module (prototype network) is active during this phase, enabling the model to leverage similar diseases' treatment profiles for zero-shot generalization.

**Note:** Fine-tuning takes ~500 epochs. For inference, skip to Section 6.

In [ ]:
# Fine-tuning (skip if loading pretrained model)
# In the paper: n_epoch=500, learning_rate=5e-4

if TRAIN_FROM_SCRATCH:
    model.finetune(
        n_epoch=500,
        learning_rate=5e-4,
        train_print_per_n=5,
        valid_per_n=20
    )
    # Save the trained model
    model.save_model('./model_ckpt')
    print("Model saved to ./model_ckpt")
else:
    print("Skipping fine-tuning — will load pretrained model in Section 6.")

---
## 6. Load Pretrained Model

Load a pretrained TxGNN model checkpoint. This restores the model config and weights, re-initializes the architecture, and sets `best_model` for inference.

The pretrained model in `TxGNNExplorer/` was trained on the **full graph** split.

In [ ]:
if not TRAIN_FROM_SCRATCH:
    # Load the pretrained model from TxGNNExplorer directory
    # This directory contains: config.pkl + model.pt
    model.load_pretrained(MODEL_DIR)
    print(f"Loaded pretrained model from {MODEL_DIR}")
    print(f"Model config: {model.config}")
    
    n_params = sum(p.numel() for p in model.best_model.parameters())
    print(f"Total parameters: {n_params:,}")

---
## 7. Retrieve ID Mappings

Build mappings between:
- **Graph indices** (integer node IDs in the DGL graph) → **Entity IDs** (e.g., DrugBank ID, MONDO ID)
- **Entity IDs** → **Human-readable names** (e.g., "Aspirin", "Type 2 Diabetes")

These mappings are essential for interpreting model predictions.

In [ ]:
# Retrieve drug/disease ID and name mappings
id_mappings = TxData_obj.retrieve_id_mapping()

id2name_drug = id_mappings['id2name_drug']
id2name_disease = id_mappings['id2name_disease']
idx2id_drug = id_mappings['idx2id_drug']
idx2id_disease = id_mappings['idx2id_disease']

# Build reverse mappings
id2idx_drug = {v: k for k, v in idx2id_drug.items()}
id2idx_disease = {v: k for k, v in idx2id_disease.items()}
name2id_disease = {v: k for k, v in id2name_disease.items()}
name2id_drug = {v: k for k, v in id2name_drug.items()}

print(f"Number of drugs:    {len(id2name_drug):,}")
print(f"Number of diseases: {len(id2name_disease):,}")

# Show a few sample mappings
sample_diseases = list(id2name_disease.items())[:5]
print("\nSample disease mappings:")
for did, dname in sample_diseases:
    print(f"  {did} → {dname}")

---
## 8. Disease-Centric Evaluation — Full Test Set

The disease-centric evaluation protocol from the paper:

For each test disease $d$:
1. Score **all** drugs against $d$ using the trained DistMult decoder
2. Rank drugs by prediction score
3. Compute metrics: AUROC, AUPRC, Recall@K, MRR, AP, enrichment over random

This evaluates the model's ability to identify therapeutic candidates from the full drug library for diseases it has never seen during training.

In [ ]:
# Initialize the evaluation module
evaluator = TxEval(model=model)

# Get all disease indices in the test set for the 'indication' relation
test_disease_idxs = evaluator.retrieve_disease_idxs_test_set('indication')
print(f"Number of test diseases (indication): {len(test_disease_idxs)}")
print(f"Sample disease indices: {test_disease_idxs[:10]}")

In [ ]:
# Run disease-centric evaluation on the full test set
# This evaluates all three relations: indication, contraindication, off-label use
# return_raw=True gives us detailed per-disease predictions, labels, and metrics

result = evaluator.eval_disease_centric(
    disease_idxs='test_set',
    show_plot=False,
    verbose=True,
    save_result=False,
    return_raw=True
)

In [ ]:
# Inspect the evaluation results structure
print("=== Result Keys ===")
for key in result.keys():
    print(f"  {key}: {type(result[key]).__name__}")
    if isinstance(result[key], dict):
        for subkey in list(result[key].keys())[:3]:
            print(f"    {subkey}: {type(result[key][subkey]).__name__}")

In [ ]:
# Extract and display summary metrics for the 'indication' relation
indication_result = result['result']['rev_indication']

# Build summary DataFrame
summary_data = []
for disease_id in indication_result['Name'].keys():
    row = {
        'Disease': indication_result['Name'][disease_id],
        'AUROC': indication_result['AUROC'][disease_id],
        'AUPRC': indication_result['AUPRC'][disease_id],
        'Recall@10': indication_result['Recall@10'][disease_id],
        'Recall@50': indication_result['Recall@50'][disease_id],
        'Recall@100': indication_result['Recall@100'][disease_id],
        '# Positives': indication_result['# of Pos'][disease_id]
    }
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
# Filter out diseases with no positive test edges (AUROC=-1)
df_summary_valid = df_summary[df_summary['AUROC'] != -1]

print(f"=== Indication Prediction — Test Set Summary ===")
print(f"Diseases evaluated: {len(df_summary_valid)}")
print(f"\nMean AUROC:     {df_summary_valid['AUROC'].mean():.4f} ± {df_summary_valid['AUROC'].std():.4f}")
print(f"Mean AUPRC:     {df_summary_valid['AUPRC'].mean():.4f} ± {df_summary_valid['AUPRC'].std():.4f}")
print(f"Mean Recall@50: {df_summary_valid['Recall@50'].mean():.4f} ± {df_summary_valid['Recall@50'].std():.4f}")
print(f"Mean Recall@100:{df_summary_valid['Recall@100'].mean():.4f} ± {df_summary_valid['Recall@100'].std():.4f}")

df_summary_valid.sort_values('AUROC', ascending=False).head(15)

In [ ]:
# Visualize performance distribution across diseases
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_summary_valid['AUROC'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df_summary_valid['AUROC'].mean(), color='red', linestyle='--', label=f"Mean: {df_summary_valid['AUROC'].mean():.3f}")
axes[0].set_xlabel('AUROC')
axes[0].set_ylabel('Count')
axes[0].set_title('AUROC Distribution (Indication)')
axes[0].legend()

axes[1].hist(df_summary_valid['AUPRC'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(df_summary_valid['AUPRC'].mean(), color='red', linestyle='--', label=f"Mean: {df_summary_valid['AUPRC'].mean():.3f}")
axes[1].set_xlabel('AUPRC')
axes[1].set_ylabel('Count')
axes[1].set_title('AUPRC Distribution (Indication)')
axes[1].legend()

axes[2].hist(df_summary_valid['Recall@50'], bins=30, edgecolor='black', alpha=0.7, color='seagreen')
axes[2].axvline(df_summary_valid['Recall@50'].mean(), color='red', linestyle='--', label=f"Mean: {df_summary_valid['Recall@50'].mean():.3f}")
axes[2].set_xlabel('Recall@50')
axes[2].set_ylabel('Count')
axes[2].set_title('Recall@50 Distribution (Indication)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 9. Inference on Specific Diseases

Given a specific disease (by index or name), rank all drugs by their predicted therapeutic potential. This is the core inference mode for drug repurposing.

In [ ]:
def lookup_disease_idx(disease_name_query, name2id=name2id_disease, id2idx=id2idx_disease):
    """Search for a disease by partial name match."""
    matches = [(name, did) for name, did in name2id.items() 
               if disease_name_query.lower() in name.lower()]
    if not matches:
        print(f"No diseases found matching '{disease_name_query}'")
        return []
    
    results = []
    for name, did in matches[:20]:
        idx = id2idx.get(did, None)
        results.append({'name': name, 'id': did, 'idx': idx})
        print(f"  {name:50s} | ID: {did:15s} | idx: {idx}")
    return results

# Example: search for diseases
print("=== Searching for 'macular' ===")
_ = lookup_disease_idx('macular')
print("\n=== Searching for 'Alzheimer' ===")
_ = lookup_disease_idx('Alzheimer')

In [ ]:
def predict_drugs_for_disease(model_obj, disease_idx, relation='indication',
                              id_mappings=None, top_k=20):
    """
    Run TxGNN inference to predict top drug candidates for a given disease.
    
    Parameters
    ----------
    model_obj : TxGNN
        Trained TxGNN model object
    disease_idx : float
        Graph index of the disease node
    relation : str
        'indication', 'contraindication', or 'off-label use'
    id_mappings : dict
        Output of TxData.retrieve_id_mapping()
    top_k : int
        Number of top drugs to return
        
    Returns
    -------
    pd.DataFrame with ranked drug predictions
    """
    import dgl
    
    device = model_obj.device
    G = model_obj.G.to(device)
    best_model = model_obj.best_model
    best_model.eval()
    
    rel_type = 'rev_' + relation
    drug_nodes = G.nodes('drug').cpu().numpy()
    n_drugs = len(drug_nodes)
    
    # Construct evaluation graph: disease → all drugs
    src = torch.tensor([int(disease_idx)] * n_drugs, dtype=torch.int64, device=device)
    dst = torch.tensor(drug_nodes, dtype=torch.int64, device=device)
    
    out = {('disease', rel_type, 'drug'): (src, dst)}
    g_eval = dgl.heterograph(out, num_nodes_dict={
        ntype: G.number_of_nodes(ntype) for ntype in G.ntypes
    }).to(device)
    
    # Run inference
    with torch.no_grad():
        _, pred_score_rel, _, _ = best_model(G, g_eval)
    
    scores = pred_score_rel[('disease', rel_type, 'drug')].reshape(-1).detach().cpu().numpy()
    
    # Build ranked results
    idx2id_drug_map = id_mappings['idx2id_drug']
    id2name_drug_map = id_mappings['id2name_drug']
    idx2id_disease_map = id_mappings['idx2id_disease']
    id2name_disease_map = id_mappings['id2name_disease']
    
    ranked_indices = np.argsort(scores)[::-1]
    
    results = []
    for rank, idx in enumerate(ranked_indices[:top_k]):
        drug_node_idx = drug_nodes[idx]
        drug_id = idx2id_drug_map.get(drug_node_idx, 'N/A')
        drug_name = id2name_drug_map.get(drug_id, 'Unknown')
        results.append({
            'Rank': rank + 1,
            'Drug': drug_name,
            'Drug ID': drug_id,
            'Score': float(scores[idx]),
            'Drug Node Idx': int(drug_node_idx)
        })
    
    disease_id = idx2id_disease_map.get(disease_idx, 'N/A')
    disease_name = id2name_disease_map.get(disease_id, 'Unknown')
    
    print(f"\n=== Top {top_k} Predicted {relation.title()}s for: {disease_name} ===")
    print(f"Disease ID: {disease_id} | Graph Index: {disease_idx}")
    print(f"{'='*70}")
    
    return pd.DataFrame(results)

In [ ]:
# Example: Predict indications for specific test diseases
# Use test_disease_idxs from the evaluation, or pick specific ones

example_disease_idx = test_disease_idxs[0]
df_preds = predict_drugs_for_disease(
    model, 
    disease_idx=example_disease_idx,
    relation='indication',
    id_mappings=id_mappings,
    top_k=20
)
df_preds

In [ ]:
# Evaluate specific diseases with the TxEval API (returns full metrics)
# You can pass a list of disease indices

specific_disease_idxs = test_disease_idxs[:3].tolist()  # First 3 test diseases

result_specific = evaluator.eval_disease_centric(
    disease_idxs=specific_disease_idxs,
    relation='indication',
    save_result=False
)

# Display results
if isinstance(result_specific, pd.DataFrame):
    display_cols = ['Name', 'AUROC', 'AUPRC', 'Recall@10', 'Recall@50', '# of Pos']
    available_cols = [c for c in display_cols if c in result_specific.columns]
    result_specific[available_cols]

---
## 10. Batch Prediction via `model.predict()`

The `predict()` method scores a given set of drug-disease pairs directly from a DataFrame. This is useful when you have a specific set of (drug, disease) candidates to evaluate.

In [ ]:
# Construct a DataFrame of candidate drug-disease pairs to score
# Here we use test edges as an example

df_test_indication = TxData_obj.df_test[
    TxData_obj.df_test.relation == 'indication'
].head(20).copy()

print(f"Scoring {len(df_test_indication)} test indication pairs...")
print(df_test_indication[['x_idx', 'relation', 'y_idx']].head())

# Get prediction scores
pred_scores = model.predict(df_test_indication)

# Extract scores for the indication relation
for etype, scores in pred_scores.items():
    if scores.numel() > 0:
        print(f"\n{etype}: {scores.shape}")
        print(f"  Scores: {torch.sigmoid(scores).detach().cpu().numpy().flatten()[:10]}")

---
## 11. Node Embedding Retrieval

Extract learned node embeddings from the trained GNN. These 100-dimensional embeddings capture the structural and functional properties of each node in the KG, and can be used for:
- Disease similarity analysis
- Drug clustering
- Downstream ML tasks

In [ ]:
# Retrieve node embeddings from the trained model
node_embeddings = model.retrieve_embedding()

print("=== Node Embeddings ===")
for ntype, emb in node_embeddings.items():
    print(f"  {ntype:25s}: shape={emb.shape}, dtype={emb.dtype}")

In [ ]:
# Compute disease-disease similarity using embeddings
from sklearn.metrics.pairwise import cosine_similarity

disease_emb = node_embeddings['disease'].numpy()
drug_emb = node_embeddings['drug'].numpy()

# Example: find diseases most similar to a query disease
def find_similar_diseases(query_idx, disease_emb, idx2id_disease, id2name_disease, top_k=10):
    """Find the most similar diseases to a query disease by cosine similarity."""
    query_vec = disease_emb[int(query_idx)].reshape(1, -1)
    sims = cosine_similarity(query_vec, disease_emb).flatten()
    top_indices = np.argsort(sims)[::-1][1:top_k+1]  # exclude self
    
    results = []
    for idx in top_indices:
        did = idx2id_disease.get(float(idx), idx2id_disease.get(idx, 'N/A'))
        dname = id2name_disease.get(did, 'Unknown')
        results.append({'Disease': dname, 'ID': did, 'Similarity': sims[idx]})
    
    return pd.DataFrame(results)

# Find diseases similar to the first test disease
query_disease_idx = test_disease_idxs[0]
query_disease_id = idx2id_disease.get(query_disease_idx, 'N/A')
query_disease_name = id2name_disease.get(query_disease_id, 'Unknown')

print(f"Diseases most similar to: {query_disease_name}")
df_similar = find_similar_diseases(query_disease_idx, disease_emb, idx2id_disease, id2name_disease)
df_similar

In [ ]:
# Visualize disease embeddings with t-SNE
from sklearn.manifold import TSNE

# Use a subset for visualization speed
n_sample = min(2000, disease_emb.shape[0])
np.random.seed(42)
sample_idx = np.random.choice(disease_emb.shape[0], n_sample, replace=False)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(disease_emb[sample_idx])

plt.figure(figsize=(10, 8))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], s=5, alpha=0.5, c='steelblue')

# Highlight test diseases
test_mask = np.isin(sample_idx, [int(x) for x in test_disease_idxs])
if test_mask.any():
    plt.scatter(emb_2d[test_mask, 0], emb_2d[test_mask, 1], 
                s=20, alpha=0.8, c='red', label='Zero-shot test diseases')

plt.title('TxGNN Disease Embeddings (t-SNE)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
plt.show()

---
## 12. Retrieve Similar Diseases (Metric Learning Prototypes)

One of TxGNN's key innovations is the **metric learning module** that retrieves similar diseases for zero-shot augmentation. We can inspect which diseases the model uses as prototypes for a given target disease.

In [ ]:
# Retrieve the similar diseases used by the prototype network
# This shows which diseases the model leverages for zero-shot transfer

try:
    similar_diseases = model.retrieve_sim_diseases(
        relation='indication',
        k=5  # top 5 similar diseases per target
    )
    print(f"Similar diseases tensor shape: {similar_diseases.shape}")
    print(f"\nPrototype disease indices (first 5 target diseases):")
    print(similar_diseases[:5])
except Exception as e:
    print(f"Note: retrieve_sim_diseases requires the metric learning module. Error: {e}")

---
## 13. GraphMask Explainability (Post-hoc Explanations)

TxGNN uses **GraphMask** to provide post-hoc explanations of its predictions. GraphMask learns binary gates on each edge of the GNN, identifying which edges are essential for a prediction while maintaining similar output.

Steps:
1. Train the GraphMask model (or load pretrained)
2. Retrieve attention/gate scores per edge
3. Find meta-paths (interpretive reasoning chains) between drug-disease pairs

In [ ]:
# Training GraphMask (skip if loading pretrained)
# This is computationally expensive — use pretrained gates if available

TRAIN_GRAPHMASK = False

if TRAIN_GRAPHMASK:
    model.train_graphmask(
        relation='indication',
        learning_rate=3e-4,
        allowance=0.005,
        epochs_per_layer=3,
        penalty_scaling=1,
        valid_per_n=20
    )
    
    # Save gates
    output = model.retrieve_save_gates('./model_ckpt')
    model.save_graphmask_model('./graphmask_model_ckpt')
    print("GraphMask trained and saved.")
else:
    print("Loading pre-computed GraphMask attention scores...")

In [ ]:
# Load pre-computed GraphMask attention gates
graphmask_path = os.path.join(MODEL_DIR, 'graphmask_output_indication.pkl')

if os.path.exists(graphmask_path):
    df_gates = pd.read_pickle(graphmask_path)
    print(f"Loaded GraphMask gates: {df_gates.shape}")
    print(f"\nColumns: {df_gates.columns.tolist()}")
    print(f"\nRelation types: {df_gates.relation.unique()}")
    print(f"\nSample edges:")
    df_gates[['x_name', 'relation', 'y_name', 'indication_layer1_att', 'indication_layer2_att']].head(10)
else:
    print(f"GraphMask file not found at {graphmask_path}")
    df_gates = None

In [ ]:
# Build a graph of important edges for meta-path finding
import networkx as nx

if df_gates is not None:
    # Filter out low-attention edges and CYP-related genes (less interpretable)
    cyp_filter = ~df_gates.x_name.str.startswith('CYP', na=False) & \
                 ~df_gates.y_name.str.startswith('CYP', na=False)
    df_gates_filtered = df_gates[cyp_filter].copy()
    
    # Build NetworkX MultiDiGraph
    G_nx = nx.MultiDiGraph()
    
    for _, row in df_gates_filtered.iterrows():
        G_nx.add_edge(
            f"{row['x_type']}:{row['x_name']}",
            f"{row['y_type']}:{row['y_name']}",
            relation=row['relation'],
            layer1_att=row['indication_layer1_att'],
            layer2_att=row['indication_layer2_att'],
            avg_att=(row['indication_layer1_att'] + row['indication_layer2_att']) / 2
        )
    
    print(f"Built explanation graph: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")

In [ ]:
def find_explanation_paths(G_nx, disease_name, drug_name, max_path_length=4, top_k=10):
    """
    Find meta-paths between a disease and drug in the explanation graph.
    Paths are scored by average attention weight.
    """
    disease_node = f"disease:{disease_name}"
    drug_node = f"drug:{drug_name}"
    
    if disease_node not in G_nx.nodes:
        print(f"Disease '{disease_name}' not found in graph")
        return pd.DataFrame()
    if drug_node not in G_nx.nodes:
        print(f"Drug '{drug_name}' not found in graph")
        return pd.DataFrame()
    
    # Find all simple paths
    undirected = G_nx.to_undirected()
    paths = list(nx.all_simple_paths(undirected, disease_node, drug_node, cutoff=max_path_length))
    
    if not paths:
        print(f"No paths found between {disease_name} and {drug_name}")
        return pd.DataFrame()
    
    # Score paths by average attention
    scored_paths = []
    for path in paths:
        att_scores = []
        relations = []
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            # Get best attention edge
            if G_nx.has_edge(u, v):
                edges = G_nx[u][v]
            elif G_nx.has_edge(v, u):
                edges = G_nx[v][u]
            else:
                continue
            best_att = max(edges[k].get('avg_att', 0) for k in edges)
            best_rel = max(edges, key=lambda k: edges[k].get('avg_att', 0))
            att_scores.append(best_att)
            relations.append(edges[best_rel].get('relation', 'unknown'))
        
        if att_scores:
            meta_path = ' → '.join([n.split(':')[0] for n in path])
            node_names = ' → '.join([n.split(':')[1] if ':' in n else n for n in path])
            scored_paths.append({
                'Meta-Path': meta_path,
                'Path': node_names,
                'Relations': ' → '.join(relations),
                'Avg Attention': np.mean(att_scores),
                'Path Length': len(path)
            })
    
    df_paths = pd.DataFrame(scored_paths)
    df_paths = df_paths.sort_values('Avg Attention', ascending=False).head(top_k)
    
    print(f"\n=== Explanation Paths: {disease_name} ↔ {drug_name} ===")
    print(f"Found {len(paths)} paths, showing top {min(top_k, len(df_paths))} by attention score")
    
    return df_paths.reset_index(drop=True)

In [ ]:
# Example: Explain a prediction
if df_gates is not None:
    # Pick a disease-drug pair from the top predictions
    # Adjust these names based on your data
    example_disease = df_summary_valid.iloc[0]['Disease'] if len(df_summary_valid) > 0 else 'macular degeneration'
    
    # Find a top-predicted drug for this disease
    try:
        disease_matches = lookup_disease_idx(example_disease)
        if disease_matches:
            ex_idx = disease_matches[0]['idx']
            df_top_drugs = predict_drugs_for_disease(
                model, ex_idx, 'indication', id_mappings, top_k=5
            )
            if len(df_top_drugs) > 0:
                top_drug_name = df_top_drugs.iloc[0]['Drug']
                df_explain = find_explanation_paths(G_nx, example_disease, top_drug_name)
                if len(df_explain) > 0:
                    display(df_explain)
    except Exception as e:
        print(f"Explanation example skipped: {e}")

---
## 14. Full Inference Pipeline — End-to-End Function

A convenience function that runs the complete inference pipeline for a disease name query: search → predict → rank → explain.

In [ ]:
def txgnn_infer(disease_query, model_obj, id_mappings, G_nx=None,
                relation='indication', top_k=20, explain=True, explain_top_n=3):
    """
    End-to-end TxGNN inference pipeline.
    
    Parameters
    ----------
    disease_query : str
        Disease name (partial match supported)
    model_obj : TxGNN
        Trained TxGNN model
    id_mappings : dict
        From TxData.retrieve_id_mapping()
    G_nx : nx.MultiDiGraph, optional
        GraphMask explanation graph
    relation : str
        'indication', 'contraindication', or 'off-label use'
    top_k : int
        Number of top drug predictions to return
    explain : bool
        Whether to generate GraphMask explanations
    explain_top_n : int
        Number of top drugs to explain
    """
    name2id = {v: k for k, v in id_mappings['id2name_disease'].items()}
    id2idx = {v: k for k, v in id_mappings['idx2id_disease'].items()}
    
    # Step 1: Find the disease
    matches = [(name, did) for name, did in name2id.items()
               if disease_query.lower() in name.lower()]
    
    if not matches:
        print(f"No disease found matching '{disease_query}'")
        return None
    
    if len(matches) > 1:
        print(f"Multiple matches found for '{disease_query}':")
        for name, did in matches[:10]:
            print(f"  - {name}")
        print(f"Using first match: {matches[0][0]}")
    
    disease_name, disease_id = matches[0]
    disease_idx = id2idx.get(disease_id)
    
    if disease_idx is None:
        print(f"Disease index not found for {disease_name}")
        return None
    
    # Step 2: Predict drugs
    df_preds = predict_drugs_for_disease(
        model_obj, disease_idx, relation, id_mappings, top_k
    )
    
    # Step 3: Explain top predictions
    explanations = {}
    if explain and G_nx is not None:
        for i in range(min(explain_top_n, len(df_preds))):
            drug_name = df_preds.iloc[i]['Drug']
            df_explain = find_explanation_paths(G_nx, disease_name, drug_name, top_k=5)
            if len(df_explain) > 0:
                explanations[drug_name] = df_explain
    
    return {
        'disease': disease_name,
        'predictions': df_preds,
        'explanations': explanations
    }

In [ ]:
# Run end-to-end inference
result_infer = txgnn_infer(
    disease_query='Alzheimer',  # Change this to any disease of interest
    model_obj=model,
    id_mappings=id_mappings,
    G_nx=G_nx if df_gates is not None else None,
    relation='indication',
    top_k=20,
    explain=True,
    explain_top_n=3
)

if result_infer:
    print("\n=== Predictions ===")
    display(result_infer['predictions'])
    
    if result_infer['explanations']:
        print("\n=== Explanations ===")
        for drug, df_exp in result_infer['explanations'].items():
            print(f"\n--- {drug} ---")
            display(df_exp)

---
## 15. Prediction Score Distribution Analysis

Analyze the distribution of prediction scores to understand model calibration and identify meaningful thresholds.

In [ ]:
# Analyze prediction score distributions for indication relation
if 'prediction' in result and 'label' in result:
    preds_all = result['prediction']['rev_indication']
    labels_all = result['label']['rev_indication']
    
    all_pos_scores = []
    all_neg_scores = []
    
    for disease_id in preds_all:
        pred = preds_all[disease_id]
        lab = labels_all[disease_id]
        for drug_id in pred:
            if lab[drug_id] == 1:
                all_pos_scores.append(pred[drug_id])
            elif lab[drug_id] == 0:
                all_neg_scores.append(pred[drug_id])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Score distribution
    axes[0].hist(all_neg_scores, bins=50, alpha=0.6, label=f'Negative (n={len(all_neg_scores):,})', 
                 density=True, color='steelblue')
    axes[0].hist(all_pos_scores, bins=50, alpha=0.6, label=f'Positive (n={len(all_pos_scores):,})', 
                 density=True, color='coral')
    axes[0].set_xlabel('Prediction Score')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Score Distribution: Positive vs Negative Pairs')
    axes[0].legend()
    
    # Score separation (box plot)
    data_box = [all_pos_scores[:5000], all_neg_scores[:5000]]  # subsample for speed
    axes[1].boxplot(data_box, labels=['Positive\n(True Indications)', 'Negative\n(Non-Indications)'])
    axes[1].set_ylabel('Prediction Score')
    axes[1].set_title('Score Separation')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPositive scores — Mean: {np.mean(all_pos_scores):.4f}, Median: {np.median(all_pos_scores):.4f}")
    print(f"Negative scores — Mean: {np.mean(all_neg_scores):.4f}, Median: {np.median(all_neg_scores):.4f}")

---
## 16. Knowledge Graph Statistics & Exploration

Inspect the underlying knowledge graph structure to understand the data that TxGNN leverages.

In [ ]:
# Knowledge graph statistics
G = model.G

print("=== Knowledge Graph Summary ===")
print(f"\nNode Types ({len(G.ntypes)}):")
for ntype in sorted(G.ntypes):
    print(f"  {ntype:30s}: {G.number_of_nodes(ntype):>8,} nodes")

print(f"\nEdge Types ({len(G.canonical_etypes)}):")
for etype in sorted(G.canonical_etypes, key=lambda x: x[1]):
    n_edges = G.number_of_edges(etype)
    if n_edges > 0:
        print(f"  ({etype[0]:20s}) --[{etype[1]:30s}]--> ({etype[2]:20s}): {n_edges:>8,} edges")

print(f"\nTotal: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

---
## Summary

This notebook demonstrated the complete TxGNN inference pipeline:

| Step | Description | Key Function |
|------|-------------|-------------|
| 1 | Data loading | `TxData(data_folder_path)` |
| 2 | Split preparation | `TxData.prepare_split(split, seed)` |
| 3 | Model initialization | `TxGNN.model_initialize(...)` |
| 4 | Pre-training | `TxGNN.pretrain(...)` |
| 5 | Fine-tuning | `TxGNN.finetune(...)` |
| 6 | Load pretrained | `TxGNN.load_pretrained(path)` |
| 7 | ID mappings | `TxData.retrieve_id_mapping()` |
| 8 | Test evaluation | `TxEval.eval_disease_centric(...)` |
| 9 | Single-disease inference | `predict_drugs_for_disease(...)` |
| 10 | Batch prediction | `TxGNN.predict(df)` |
| 11 | Embedding retrieval | `TxGNN.retrieve_embedding()` |
| 12 | Prototype diseases | `TxGNN.retrieve_sim_diseases(...)` |
| 13 | GraphMask XAI | `TxGNN.train_graphmask(...)` |
| 14 | End-to-end inference | `txgnn_infer(...)` |

### Key References
- **Paper**: Huang et al., "Zero-shot Prediction of Therapeutic Use with Geometric Deep Learning" (2023)
- **Code**: [github.com/mims-harvard/TxGNN](https://github.com/mims-harvard/TxGNN)
- **Explorer**: [txgnn.org](http://txgnn.org)